# Experimento con el LLM: ¿respeta las reglas de producción?

**Responsable:** Persona 5

Se envían al LLM las reglas de nuestra gramática (`spanish/grammar.py`) y se le pide aceptar o
rechazar cada oración del set de prueba y dar su árbol.

| Versión | Prompt | Salida |
| --- | --- | --- |
| zero-shot | solo las reglas | `results/llm_zero_shot.jsonl` |
| few-shot | reglas + ejemplos resueltos | `results/llm_few_shot.jsonl` |

Todos los prompts y respuestas quedan en `results/logs/llm_log.jsonl`.

**Claves de API:** nunca en el notebook. Van en un archivo `.env` en la raíz del repositorio
(ya está en `.gitignore`), por ejemplo `ANTHROPIC_API_KEY=...`. Con `PROVIDER = "manual"` no hace
falta ninguna clave: el notebook muestra el prompt y se pega la respuesta de cualquier chat.

Preguntas a responder (van al informe):

1. ¿El LLM respeta las reglas o se desvía hacia su conocimiento del español?
2. ¿Dónde acierta y dónde falla?
3. ¿El few-shot mejora la precisión?

In [ ]:
# Configuración: el notebook vive en notebooks/, el código del proyecto en la raíz.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evaluation.formats import Prediction, load_test_set, save_predictions, results_path

items = load_test_set()
print(f"{len(items)} oraciones en el set de prueba")

In [ ]:
# Configuración del experimento. Anotar en el informe el modelo exacto y la fecha.
PROVIDER = "manual"            # "manual" | "anthropic" | "openai" | "gemini"
OPTIONS = {"temperature": 0}   # TODO(P5): agregar model="..." según el proveedor
N_EXAMPLES = 4                 # ejemplos del few-shot

In [ ]:
from spanish.grammar import as_bnf

print(as_bnf())

## 1. Zero-shot

In [ ]:
from experiments import prompts

print(prompts.zero_shot(items[0].sentence))

In [ ]:
from experiments.llm_client import ask_llm


def to_prediction(item, answer: str, system: str) -> Prediction:
    parsed = prompts.parse_answer(answer)
    return Prediction(
        id=item.id,
        system=system,
        accepted=bool(parsed["aceptada"]),
        tree=parsed["arbol"],
        error_type=parsed["tipo_error"],
        detail=parsed["explicacion"] or "",
        extra={"formato_valido": parsed["formato_valido"], "respuesta": answer},
    )


zero_shot = []
for item in items:
    answer = ask_llm(
        prompts.zero_shot(item.sentence),
        provider=PROVIDER,
        metadata={"id": item.id, "modo": "zero_shot"},
        **OPTIONS,
    )
    zero_shot.append(to_prediction(item, answer, "llm_zero_shot"))

print("Guardado en", save_predictions("llm_zero_shot", zero_shot))

## 2. Few-shot

Los ejemplos salen del set de prueba, pero **nunca** incluyen la oración que se pregunta.

In [ ]:
def pick_examples(target, n: int = N_EXAMPLES):
    """Ejemplos variados: aceptadas y rechazadas, de distintos niveles.

    TODO(P5): decidir el criterio y mantenerlo fijo para todo el experimento
    (por ejemplo, un conjunto fijo de ejemplos que NO se evalúan).
    """
    others = [i for i in items if i.id != target.id]
    accepted = [i for i in others if i.covered]
    rejected = [i for i in others if not i.covered]
    half = n // 2
    return accepted[:n - half] + rejected[:half]


print(prompts.few_shot(items[0].sentence, pick_examples(items[0])))

In [ ]:
few_shot = []
for item in items:
    answer = ask_llm(
        prompts.few_shot(item.sentence, pick_examples(item)),
        provider=PROVIDER,
        metadata={"id": item.id, "modo": "few_shot"},
        **OPTIONS,
    )
    few_shot.append(to_prediction(item, answer, "llm_few_shot"))

print("Guardado en", save_predictions("llm_few_shot", few_shot))

## 3. Análisis

Las métricas completas las calcula la Persona 4 (`evaluation/metrics.py`). Aquí van las
preguntas propias del experimento.

In [ ]:
from evaluation.formats import load_predictions

by_id = {i.id: i for i in items}

for system in ("llm_zero_shot", "llm_few_shot"):
    if not results_path(system).exists():
        continue
    preds = load_predictions(system)
    valid = sum(p.extra.get("formato_valido", False) for p in preds)
    right = sum(p.accepted == by_id[p.id].covered for p in preds)
    # Desviación: acepta español correcto que la gramática NO cubre (sigue su conocimiento, no las reglas).
    drift = [p.id for p in preds if p.accepted and by_id[p.id].grammatical and not by_id[p.id].covered]
    print(f"{system}: formato válido {valid}/{len(preds)}, decisión correcta {right}/{len(preds)}")
    print(f"  se desvía hacia el español en: {drift or 'ninguna'}")

### Conclusiones (para el informe)

TODO(P5):

1. ¿Respeta las reglas o se desvía? Usar la lista de desviaciones de arriba y ejemplos del log.
2. ¿Dónde acierta y dónde falla? Por nivel y por tipo de error.
3. ¿El few-shot mejora la precisión? Comparar las dos filas.
4. Para la comparación de enfoques: robustez, ambigüedad, escalabilidad y costo (tokens, tiempo y dinero; el log tiene los segundos por llamada).